# ShopDesk, Module 3 Section 2 Lab 2: Plan Mode, Direct Execution, and Explore

A beginner-friendly notebook on **matching the execution mode to the task**. A big multi-file
refactor deserves **plan mode** (think first, get approval, then act); a one-line bug fix deserves
**direct execution**; and codebase discovery belongs in the **Explore** subagent so it does not flood
your main context. Pure-Python cells make the choices and the savings concrete offline; live
**Claude Agent SDK** runs show plan mode proposing without editing and direct execution making the
change. Runs **Sonnet** (`claude-sonnet-4-6`) through your **Anthropic API key**.

## The real-world scenario

ShopDesk has two tasks waiting: reshape the refund code across several modules, and fix a small bug in
one status label. Treating both the same way wastes effort. The refactor needs a reviewed plan before
any code changes; the bug fix just needs doing. And before either, understanding the current code is a
job for Explore, which investigates in isolation and returns a summary.

The question this lab answers: **when do you plan, when do you just execute, and how does Explore keep
discovery from eating your context?**

## Objectives

- Choose **plan mode** vs **direct execution** from the shape of the task.
- See plan mode **propose without editing**, and direct execution **make the change**.
- Use the **Explore** subagent to isolate discovery and measure the context it saves.

## What you'll observe

- A simple classifier recommends direct for small changes and plan for multi-file or architectural ones.
- Live, plan mode leaves every file byte-for-byte unchanged; direct execution edits the target file.
- Explore returns a short summary instead of loading many files into your main context.

## How to run

Run top to bottom. The repo, the classifier, and the Explore measurement run anywhere. The two live
cells call Claude, so paste a real key into **Setup 2/3** and re-run from the top; otherwise they skip.
**Node.js 18+** must be installed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages. The live cells use the **Agent SDK** to drive plan mode and
direct execution; it needs Node.js 18+.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports, the model, the `RUN_LIVE` switch, `run_async()`, and a `snapshot()` helper
that hashes every file so we can prove which mode changed the repo.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, an async runner, a snapshot =====
import os                                       # filesystem paths for the sample repo
import sys                                       # detect Windows (special event loop)
import hashlib                                  # hash files to detect changes
from pathlib import Path                          # walk the repo
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cells will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

def snapshot(root):                              # map every file -> its content hash
    return {str(p.relative_to(root)): hashlib.md5(p.read_bytes()).hexdigest()
            for p in sorted(Path(root).rglob("*")) if p.is_file()}

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** writes a small **ShopDesk sample repo**: the refund path plus a status helper that
has a small bug (it raises on an unknown status code). The refactor task and the bug fix both act on
these files.

In [ ]:
# ===== SETUP 3/3 - create the sample repo (refund path + a small bug) =====
import textwrap                                    # keeps the embedded file bodies readable
REPO = os.path.join(os.getcwd(), "shopdesk_modes") # the sample codebase root

FILES = {
    "shopdesk/orders.py": textwrap.dedent("""\
        STATUS_NAMES = {1: "processing", 2: "shipped", 3: "delivered"}

        def status_label(code):
            return STATUS_NAMES[code]
        """),
    "shopdesk/refunds.py": textwrap.dedent("""\
        from shopdesk.orders import status_label

        REFUND_WINDOW_DAYS = 30

        def is_refundable(order):
            return order["delivered_days_ago"] <= REFUND_WINDOW_DAYS

        def process_refund(order):
            return "refunded" if is_refundable(order) else "refused"
        """),
    "shopdesk/shipping.py": 'def get_tracking(order_id):\n    return "https://track.example/" + order_id\n',
}
for rel, content in FILES.items():                 # write every file
    path = os.path.join(REPO, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)
print("created repo at", REPO, "with", len(FILES), "files")

### Three ways to work

- **Direct execution:** clear scope, obvious approach, small change. Just do it.
- **Plan mode:** multiple approaches, many files, or an architectural decision. It is a read-only gate:
  Claude proposes a plan and waits for approval before touching anything.
- **Explore subagent:** you need to understand the code first. Explore is read-only (Read, Grep, Glob),
  runs on a fast model, and returns a **summary** so the raw file contents never enter your main context.

The strong pattern chains them: **Explore to understand, plan to design, execute to implement.**

---

### Lab objective - match the mode to the task

**What you build:** a classifier that recommends a mode, an Explore savings measurement, and two live
runs that show plan mode proposing without editing and direct execution making the change.

**Why it helps you build real solutions:** planning a one-line fix is overhead, and executing a
ten-file refactor blind is risky. Picking the right mode is what keeps changes both safe and fast.

**How you'll see it:** the classifier splits tasks correctly, Explore returns a fraction of the tokens,
and the repo snapshot proves which mode changed files.

**This cell:** a simple **mode classifier**. It looks for signals of scope (refactor, redesign,
across multiple files) and recommends **plan** when it finds them, **direct** otherwise. This mirrors
the judgment you make before starting a task.

In [ ]:
# ===== choose plan vs direct from the task shape =====
def choose_mode(task, files_touched=1):            # task text + rough file count -> (mode, why)
    t = task.lower()
    signals = [w for w in ["refactor", "redesign", "architecture", "migrate",
                           "across", "multiple", "several", "restructure", "split"] if w in t]
    if files_touched >= 3 or signals:               #   broad scope -> plan first
        return "plan", f"multi-file or architectural (signals={signals or files_touched})"
    return "direct", "small, well-scoped change"

tasks = [
    ("Fix the typo in the refused message", 1),
    ("Refactor the refund logic across orders, refunds, and shipping", 3),
    ("Change REFUND_WINDOW_DAYS from 30 to 45", 1),
    ("Redesign the status system to support partial refunds", 2),
]
for task, n in tasks:                              # recommend a mode for each
    mode, why = choose_mode(task, n)
    print(f"  [{mode:6}] {task}  ({why})")

**This cell:** the **Explore** payoff. If discovery read every file straight into your main
conversation, that is the full byte cost; Explore instead returns a short summary. We compare the two so
the context saving is a number, not a slogan.

In [ ]:
# ===== Explore: summary in the main context vs reading everything inline =====
repo_bytes = sum(len(p.read_text()) for p in Path(REPO).rglob("*.py"))   # all source bytes
inline_tokens = repo_bytes // 4                     # approx tokens if read into main context

explore_summary = ("Refund path: process_refund calls is_refundable, which checks "
                   "REFUND_WINDOW_DAYS (30). status_label raises on unknown codes.")   # a real 2-line summary
explore_summary_tokens = len(explore_summary) // 4  # tokens of the actual summary text

print("read everything into main context (approx tokens):", inline_tokens)
print("Explore returns this summary (approx tokens)      :", explore_summary_tokens)
print(f"main context saved on this toy repo: {1 - explore_summary_tokens / inline_tokens:.0%}")
print("on a real repo (dozens of files) the inline cost is far larger, so the saving grows")

**This cell:** the live **plan mode** run. We set `permission_mode="plan"`, snapshot the repo, ask
for the multi-file refactor, then snapshot again. Plan mode is a read-only gate, so it should propose a
plan and change **nothing**; the snapshot comparison proves it.

In [ ]:
# ===== live: plan mode proposes without editing =====
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock, ToolUseBlock

PLAN_OPTS = ClaudeAgentOptions(                    # a read-only planning gate over the repo
    model=MODEL, cwd=REPO, permission_mode="plan",
    allowed_tools=["Read", "Grep", "Glob"])         # discovery tools only

async def plan(prompt):                             # stream text + tool calls from the plan run
    async for m in query(prompt=prompt, options=PLAN_OPTS):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock): print("  ->", b.name)
                elif isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:200])

if RUN_LIVE:                                        # needs a real key (and Node.js 18+)
    before = snapshot(REPO)                          # hashes before
    run_async(lambda: plan("Refactor the refund logic across orders.py, refunds.py, and shipping.py into a cleaner structure."))
    after = snapshot(REPO)                            # hashes after
    changed = [f for f in before if before[f] != after.get(f)]
    print("\nfiles changed by plan mode:", changed, "(expected: none)")
else:
    print("[skipped] expected: Claude proposes a refactor plan and edits NO files (snapshot unchanged).")

**This cell:** the live **direct execution** run for the small, well-scoped fix. We set
`permission_mode="acceptEdits"` so the edit applies without a prompt, snapshot around it, and report which
file changed. This is the mode for a change that needs no planning.

In [ ]:
# ===== live: direct execution makes the small change =====
EDIT_OPTS = ClaudeAgentOptions(                    # allow edits for a small, clear fix
    model=MODEL, cwd=REPO, permission_mode="acceptEdits",
    allowed_tools=["Read", "Grep", "Edit"])         # read then edit

async def direct(prompt):                           # stream text + tool calls from the direct run
    async for m in query(prompt=prompt, options=EDIT_OPTS):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock): print("  ->", b.name)
                elif isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:160])

if RUN_LIVE:                                        # needs a real key (and Node.js 18+)
    before = snapshot(REPO)                          # hashes before
    run_async(lambda: direct("In shopdesk/refunds.py change REFUND_WINDOW_DAYS from 30 to 45. Read the file, then Edit."))
    after = snapshot(REPO)                            # hashes after
    changed = [f for f in before if before[f] != after.get(f)]
    print("\nfiles changed by direct execution:", changed, "(expected: refunds.py)")
else:
    print("[skipped] expected: Claude Reads then Edits refunds.py; the snapshot shows refunds.py changed.")

**In the real Claude Code CLI** (this is reference, not run here):

```text
# cycle permission modes with shift+tab; plan mode is a read-only gate
shift+tab            # switch into plan mode, then approve to execute
/context             # see how much context an Explore pass saved
```

Explore usually runs automatically when you ask a broad discovery question; you can also force it from a
skill with `context: fork` and `agent: Explore`.

| anti-pattern | what to do instead |
|---|---|
| plan mode for a one-line fix | use direct execution; planning is overhead here |
| execute a ten-file refactor blind | use plan mode, review the plan, then execute |
| read 40 files into your main chat to explore | delegate to Explore; take the summary |
| skip discovery and guess the structure | Explore first, then plan, then execute |

**Lesson:** there is no single right mode, only a right match. **Direct** for small clear changes,
**plan** for multi-file or architectural work where being wrong is costly, and **Explore** to investigate
without draining your context. The strongest workflow chains them: Explore to understand, plan to design,
execute to implement.

---

## Recap - execution modes

| Mode | Use when | Effect |
|---|---|---|
| Direct | small, clear, single change | edits immediately |
| Plan | many files, architectural, several approaches | proposes, waits for approval, edits nothing yet |
| Explore | you must understand the code first | reads in isolation, returns a summary |

One principle to carry forward: **investigate in isolation, plan when the cost of being wrong is high,
and execute directly when it is not.** To run live, paste a real key into **Setup 2/3** and re-run from
the top. Then try it: give plan mode a task and confirm the snapshot stays unchanged until you approve.